# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY') 

In [2]:
from langchain_community.document_loaders import CSVLoader
loader_csv = CSVLoader(file_path="data/procurement_sample_500.csv", encoding="utf-8-sig")
data_csv = loader_csv.load()

### Example 1

#### Create our QandA application

In [3]:
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.llms import OpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.document_loaders import CSVLoader, TextLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch
from langchain.chains import LLMChain


In [4]:
import pandas as pd

df = pd.read_csv("https://assets.gov.ie/static/documents/c41c3a86/Public_Procurement_Opendata_Dataset.csv")
df.head(500).to_csv("data/procurement_sample_500.csv", index=False)

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18140\2356988260.py:3: DtypeWarning: Columns (0: Contracting Authority, 1: Tender Name, 2: Notice Published Date, 3: Directive, 4: Competition Type, 5: Main Cpv Code Description, 6: Additional CPV Codes on CFT, 7: Spend Category, 8: Contract Type, 9: Threshold Level, 10: Procedure, 11: Tender Submission Deadline, 12: Evaluation Type, 13: Cancelled Date, 14: Award Published, 15: Awarded Suppliers, 16: TED Notice Link, 17: TED CAN Link, 18: Platform) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("https://assets.gov.ie/static/documents/c41c3a86/Public_Procurement_Opendata_Dataset.csv")


In [5]:
# !pip install --upgrade --force-reinstall sentence-transformers

In [6]:
from langchain_community.document_loaders import CSVLoader

loader_csv = CSVLoader(file_path="data/procurement_sample_500.csv", encoding="utf-8-sig")

In [7]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs = {'device': 'cpu'})
).from_loaders([loader_csv])

In [8]:
llm = ChatOpenAI(temperature = 0.0)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(search_kwargs={"k": 3}), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

#### Coming up with test datapoints

In [9]:
data_csv[0]

Document(metadata={'source': 'data/procurement_sample_500.csv', 'row': 0}, page_content='Tender ID: 70917.0\nContracting Authority: Department of Housing Local Government and Heritage\nTender Name: Register of Electors Registration Forms (RFA)\nNotice Published Date: 02/01/2013\nDirective: \nCompetition Type: Bespoke\nMain Cpv Code: 22821000.0\nMain Cpv Code Description: Electoral forms.\nAdditional CPV Codes on CFT: 2,282,100,079,810,000,000,000,000,000,000\nSpend Category: Marketing, Print and Stationery\nContract Type: Supplies\nThreshold Level: National\nProcedure: Open Procedure\nTender Submission Deadline: 24/01/2013\nEvaluation Type: \nNotice Estimated Value: \nContract Duration (Months): \nCancelled Date: \nAward Published: 31/01/2013\nAwarded Value: \nNo of Bids Received: \nNo of SMEs Bids Received: \nAwarded Suppliers: \nNo of Awarded SMEs: \nTED Notice Link: \nTED CAN Link: \nPlatform: EUS Platform')

In [10]:
data_csv[10]

Document(metadata={'source': 'data/procurement_sample_500.csv', 'row': 10}, page_content='Tender ID: 71109.0\nContracting Authority: Meenkilly National School\nTender Name: To construct a classroom and resource extension to existing National School\nNotice Published Date: 03/01/2013\nDirective: \nCompetition Type: Bespoke\nMain Cpv Code: 71000000.0\nMain Cpv Code Description: Architectural, construction, engineering and inspection services.\nAdditional CPV Codes on CFT: 710,000,004,500,000\nSpend Category: Professional Services\nContract Type: Works\nThreshold Level: National\nProcedure: Restricted Procedure\nTender Submission Deadline: 14/01/2013\nEvaluation Type: \nNotice Estimated Value: \nContract Duration (Months): \nCancelled Date: \nAward Published: \nAwarded Value: \nNo of Bids Received: 8.0\nNo of SMEs Bids Received: 6.0\nAwarded Suppliers: \nNo of Awarded SMEs: \nTED Notice Link: \nTED CAN Link: \nPlatform: EUS Platform')

In [11]:
data_csv[11]

Document(metadata={'source': 'data/procurement_sample_500.csv', 'row': 11}, page_content='Tender ID: 71100.0\nContracting Authority: Ahane National School\nTender Name: One Classroom Extension to Ahane NS\nNotice Published Date: 03/01/2013\nDirective: \nCompetition Type: Bespoke\nMain Cpv Code: 45000000.0\nMain Cpv Code Description: Construction work.\nAdditional CPV Codes on CFT: 45,000,000,452,000,000,000,000,000,000,000,000,000,000,000,000\nSpend Category: Minor Building Works and Civils\nContract Type: Works\nThreshold Level: National\nProcedure: Open Procedure\nTender Submission Deadline: 24/01/2013\nEvaluation Type: \nNotice Estimated Value: \nContract Duration (Months): \nCancelled Date: \nAward Published: \nAwarded Value: \nNo of Bids Received: 1.0\nNo of SMEs Bids Received: 0.0\nAwarded Suppliers: \nNo of Awarded SMEs: \nTED Notice Link: \nTED CAN Link: \nPlatform: EUS Platform')

#### Hard-coded examples

In [12]:
from langchain.prompts import PromptTemplate

In [13]:
from langchain.prompts import PromptTemplate
from langchain.schema import BaseOutputParser
from pydantic import BaseModel, Field

examples = [
    {
        "query": "Which contracting authority issued Tender ID 70917?",
        "answer": "Department of Housing, Local Government and Heritage"
    },
    {
        "query": "Which contracting authority issued Tender ID 71109?",
        "answer": "Meenkilly National School"
    }
]

# Define the prompt template
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Which contracting authority issued Tender ID 70917?\n"
             "   Answer: Department of Housing, Local Government and Heritage\n"
             "2. Query: Which contracting authority issued Tender ID 71109?\n"
             "   Answer: Meenkilly National School\n"
             "Query: {query}\n"
             "Answer:"
)

# Define the output model
class Answer(BaseModel):
    answer: str = Field(description="The answer to the query")

# Create the output parser
class AnswerOutputParser(BaseOutputParser):
    def parse(self, text: str) -> Answer:
        # Split the response to get the answer
        answer = text.strip().split("Answer:")[-1].strip()
        return Answer(answer=answer)

# Initialize the LLM
# llm = OpenAI()
llm = ChatOpenAI()

# Create the LLMChain
llm_chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=AnswerOutputParser()
)

# Example query
query = "Which contracting authority issued Tender ID 70917?"

# Run the chain
result = llm_chain.run({"query": query})

# Print the result
print(result)


C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18140\314718716.py:44: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  llm_chain = LLMChain(
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_18140\314718716.py:54: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  result = llm_chain.run({"query": query})


answer='Department of Housing, Local Government and Heritage'


#### LLM-Generated examples

In [14]:
from langchain.evaluation.qa import QAGenerateChain

In [15]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI())

In [16]:
llm_chain = LLMChain(llm=llm, prompt=prompt_template)

In [17]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data_csv[:5]]
)

c:\Users\Gebruiker\Desktop\Backup\.venv\langchain-v0.2.x\Lib\site-packages\langchain\chains\llm.py:366: UserWarning: The apply_and_parse method is deprecated, instead pass an output parser directly to LLMChain.
  warnings.warn(


In [18]:
new_examples[0]

{'qa_pairs': {'query': 'What is the Contracting Authority for Tender ID 70917.0?',
  'answer': 'The Contracting Authority for Tender ID 70917.0 is the Department of Housing Local Government and Heritage.'}}

In [19]:
data_csv[0]

Document(metadata={'source': 'data/procurement_sample_500.csv', 'row': 0}, page_content='Tender ID: 70917.0\nContracting Authority: Department of Housing Local Government and Heritage\nTender Name: Register of Electors Registration Forms (RFA)\nNotice Published Date: 02/01/2013\nDirective: \nCompetition Type: Bespoke\nMain Cpv Code: 22821000.0\nMain Cpv Code Description: Electoral forms.\nAdditional CPV Codes on CFT: 2,282,100,079,810,000,000,000,000,000,000\nSpend Category: Marketing, Print and Stationery\nContract Type: Supplies\nThreshold Level: National\nProcedure: Open Procedure\nTender Submission Deadline: 24/01/2013\nEvaluation Type: \nNotice Estimated Value: \nContract Duration (Months): \nCancelled Date: \nAward Published: 31/01/2013\nAwarded Value: \nNo of Bids Received: \nNo of SMEs Bids Received: \nAwarded Suppliers: \nNo of Awarded SMEs: \nTED Notice Link: \nTED CAN Link: \nPlatform: EUS Platform')

In [20]:
d_flattened = [data['qa_pairs'] for data in new_examples]
d_flattened

[{'query': 'What is the Contracting Authority for Tender ID 70917.0?',
  'answer': 'The Contracting Authority for Tender ID 70917.0 is the Department of Housing Local Government and Heritage.'},
 {'query': 'What is the Tender ID for the procurement project described in the document?',
  'answer': 'The Tender ID is 71011.0.'},
 {'query': 'What is the Tender ID for the procurement opportunity described in the document?',
  'answer': 'The Tender ID is 71080.0.'},
 {'query': 'What was the Tender ID for the IP Telephony System - NEC contract with the Institute of Public Administration?',
  'answer': 'The Tender ID for the IP Telephony System - NEC contract with the Institute of Public Administration was 71098.0.'},
 {'query': 'When was the Notice Published Date for the tender related to the Demolition and excavation of existing house and Construction of Self Catering Cottage project?',
  'answer': 'The Notice Published Date for the tender was 02/01/2013.'}]

#### Combine examples

In [21]:
# examples += new_example
examples += d_flattened

In [22]:
examples[0]

{'query': 'Which contracting authority issued Tender ID 70917?',
 'answer': 'Department of Housing, Local Government and Heritage'}

In [23]:
qa.invoke(examples[0]["query"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'query': 'Which contracting authority issued Tender ID 70917?',
 'result': "I don't have information about Tender ID 70917 in the provided context."}

### Manual Evaluation - Fun part

In [24]:
import langchain
langchain.debug = True

In [25]:
qa.invoke(examples[0]["query"])

[chain/start] [chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Which contracting authority issued Tender ID 70917?"
}
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [chain:RetrievalQA > chain:StuffDocumentsChain > chain:LLMChain] Entering Chain run with input:
{
  "question": "Which contracting authority issued Tender ID 70917?",
  "context": "Tender ID: 70775.0\nContracting Authority: Department of Justice Home Affairs and Migration\nTender Name: Request for Tenders for the provision of Legal Services to the Equality Authority\nNotice Published Date: 14/01/2013\nDirective: \nCompetition Type: Bespoke\nMain Cpv Code: 79100000.0\nMain Cpv Code Description: Legal services.\nAdditional CPV Codes on CFT: 791,000,007,911,100\nSpend Category: Managed Services\nContract Type: Services\nThreshold Level: National\nProcedure: Open Procedure\nTender Submission Deadline: 04/02/2013\nEvaluation Type: \nNotice Estim

{'query': 'Which contracting authority issued Tender ID 70917?',
 'result': "I don't have information about Tender ID 70917 in the provided context."}

In [26]:
# Turn off the debug mode
langchain.debug = False

### LLM assisted evaluation

In [27]:
examples += d_flattened

In [28]:
examples

[{'query': 'Which contracting authority issued Tender ID 70917?',
  'answer': 'Department of Housing, Local Government and Heritage'},
 {'query': 'Which contracting authority issued Tender ID 71109?',
  'answer': 'Meenkilly National School'},
 {'query': 'What is the Contracting Authority for Tender ID 70917.0?',
  'answer': 'The Contracting Authority for Tender ID 70917.0 is the Department of Housing Local Government and Heritage.'},
 {'query': 'What is the Tender ID for the procurement project described in the document?',
  'answer': 'The Tender ID is 71011.0.'},
 {'query': 'What is the Tender ID for the procurement opportunity described in the document?',
  'answer': 'The Tender ID is 71080.0.'},
 {'query': 'What was the Tender ID for the IP Telephony System - NEC contract with the Institute of Public Administration?',
  'answer': 'The Tender ID for the IP Telephony System - NEC contract with the Institute of Public Administration was 71098.0.'},
 {'query': 'When was the Notice Publi

In [29]:
predictions = qa.batch(examples)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneTyp


> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.

> Finished chain.


In [30]:
predictions

[{'query': 'Which contracting authority issued Tender ID 70917?',
  'answer': 'Department of Housing, Local Government and Heritage',
  'result': "I don't have information about Tender ID 70917 in the provided context."},
 {'query': 'Which contracting authority issued Tender ID 71109?',
  'answer': 'Meenkilly National School',
  'result': "I don't have information about Tender ID 71109 in the provided context."},
 {'query': 'What is the Contracting Authority for Tender ID 70917.0?',
  'answer': 'The Contracting Authority for Tender ID 70917.0 is the Department of Housing Local Government and Heritage.',
  'result': "I don't have information on Tender ID 70917.0 to provide you with the Contracting Authority."},
 {'query': 'What is the Tender ID for the procurement project described in the document?',
  'answer': 'The Tender ID is 71011.0.',
  'result': 'The Tender IDs for the procurement projects described in the document are as follows:\n1. Tender ID: 71918.0\n2. Tender ID: 71435.0\n3.

In [31]:
from langchain.evaluation.qa import QAEvalChain

In [32]:
llm = ChatOpenAI(temperature=0)
eval_chain = QAEvalChain.from_llm(llm)

In [33]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [34]:
graded_outputs

[{'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'INCORRECT'},
 {'results': 'CORRECT'},
 {'results': 'CORRECT'}]

In [35]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    # print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

Example 0:
Question: Which contracting authority issued Tender ID 70917?
Real Answer: Department of Housing, Local Government and Heritage
Predicted Answer: I don't have information about Tender ID 70917 in the provided context.

Example 1:
Question: Which contracting authority issued Tender ID 71109?
Real Answer: Meenkilly National School
Predicted Answer: I don't have information about Tender ID 71109 in the provided context.

Example 2:
Question: What is the Contracting Authority for Tender ID 70917.0?
Real Answer: The Contracting Authority for Tender ID 70917.0 is the Department of Housing Local Government and Heritage.
Predicted Answer: I don't have information on Tender ID 70917.0 to provide you with the Contracting Authority.

Example 3:
Question: What is the Tender ID for the procurement project described in the document?
Real Answer: The Tender ID is 71011.0.
Predicted Answer: The Tender IDs for the procurement projects described in the document are as follows:
1. Tender ID: 7

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [38]:
pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings

loader = WebBaseLoader("https://en.wikipedia.org/wiki/Public_procurement")

index = VectorstoreIndexCreator(
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})
).from_loaders([loader])

llm = ChatOpenAI(temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=index.vectorstore.as_retriever(),
    return_source_documents=True,
)

USER_AGENT environment variable not set, consider setting it to identify your requests.
c:\Users\Gebruiker\Desktop\Backup\.venv\langchain-v0.2.x\Lib\site-packages\langchain\indexes\vectorstore.py:127: UserWarning: Using InMemoryVectorStore as the default vectorstore.This memory store won't persist data. You should explicitlyspecify a vectorstore when using VectorstoreIndexCreator
  warnings.warn(


In [43]:
# testing it out

question = "what is public procurement?"
result = qa_chain.invoke({"query": question})
result["result"]

'Public procurement, also known as government procurement, is the process by which governments purchase goods, works (such as construction), or services from the private sector. It involves the selection of capable organizations to provide the desired goods or services to the taxpayers. Public procurement is a significant part of the global economy, with a substantial amount of money being spent on it annually. It is regulated by laws in most countries to prevent fraud, waste, corruption, or local protectionism. The process involves competition among private sector organizations to win contracts, which can lead to innovation, economic growth, and cost-effective services.'

In [44]:
result

{'query': 'what is public procurement?',
 'result': 'Public procurement, also known as government procurement, is the process by which governments purchase goods, works (such as construction), or services from the private sector. It involves the selection of capable organizations to provide the desired goods or services to the taxpayers. Public procurement is a significant part of the global economy, with a substantial amount of money being spent on it annually. It is regulated by laws in most countries to prevent fraud, waste, corruption, or local protectionism. The process involves competition among private sector organizations to win contracts, which can lead to innovation, economic growth, and cost-effective services.',
 'source_documents': [Document(id='34a6f686-d241-4b32-b3c3-0c5b3d1950bf', metadata={'source': 'https://en.wikipedia.org/wiki/Public_procurement', 'title': 'Government procurement - Wikipedia', 'language': 'en'}, page_content='Public procurement is based on the idea 

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [47]:
eval_questions = [
    "What is government/public procurement?",
    "What percentage of GDP did public procurement account for in OECD countries in 2019?",
    "What is the purpose of the Agreement on Government Procurement (GPA)?",
    "What is procurement fraud, and what is an example of it?",
    "What are the five procurement procedures used in EU government procurement?"
]

eval_answers = [
    "Government procurement or public procurement is the purchase of goods, works, or services by the state, such as by a government agency or a state-owned enterprise.",
    "In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.",
    "The GPA is a plurilateral international treaty under the World Trade Organization that governs government procurement, aiming to prevent fraud, waste, corruption, and local protectionism.",
    "Procurement fraud is dishonestly obtaining an advantage, avoiding an obligation, or causing a loss to public property during the procurement process. An example is a kickback, where a dishonest agent of the supplier pays a dishonest agent of the purchaser to select the supplier's bid, often at an inflated price.",
    "The five procedures are: open procedure, restricted procedure, negotiated procedure, competitive dialogue, and electronic auctions."
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [48]:
examples

[{'query': 'What is government/public procurement?',
  'ground_truths': ['Government procurement or public procurement is the purchase of goods, works, or services by the state, such as by a government agency or a state-owned enterprise.']},
 {'query': 'What percentage of GDP did public procurement account for in OECD countries in 2019?',
  'ground_truths': ['In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.']},
 {'query': 'What is the purpose of the Agreement on Government Procurement (GPA)?',
  'ground_truths': ['The GPA is a plurilateral international treaty under the World Trade Organization that governs government procurement, aiming to prevent fraud, waste, corruption, and local protectionism.']},
 {'query': 'What is procurement fraud, and what is an example of it?',
  'ground_truths': ["Procurement fraud is dishonestly obtaining an advantage, avoiding an obligation, or causing a loss to public property during the procurement process. An exampl

#### Introducing RagasEvaluatorChain

`RagasEvaluatorChain` creates a wrapper around the metrics ragas provides (documented [here](https://github.com/explodinggradients/ragas/blob/main/docs/metrics.md)), making it easier to run these evaluation with langchain and langsmith.

The evaluator chain has the following APIs

- `__call__()`: call the `RagasEvaluatorChain` directly on the result of a QA chain.
- `evaluate()`: evaluate on a list of examples (with the input queries) and predictions (outputs from the QA chain). 
- `evaluate_run()`: method implemented that is called by langsmith evaluators to evaluate langsmith datasets.

lets see each of them in action to learn more.

In [49]:
result = qa_chain.invoke({"query": eval_questions[1]})
result["result"]

'In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.'

In [58]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        if old_key == "source_documents":
            result_updated[new_key] = [doc.page_content for doc in result[old_key]]
        else:
            result_updated[new_key] = result[old_key]

In [59]:
result_updated

{'question': 'What percentage of GDP did public procurement account for in OECD countries in 2019?',
 'answer': 'In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.',
 'contexts': ['Government procurement or public procurement is the purchase of goods, works (construction), or services by the state, such as by a government agency or a state-owned enterprise.[1][2][3] In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.[4][5] In 2021, the World Bank Group estimated that public procurement made up about 15% of global GDP.[6] Therefore, government procurement accounts for a substantial part of the global economy.',
  "United Kingdom[edit]\nMain article: Government procurement in the United Kingdom\nAt around £290 billion every year, public sector procurement accounts for around a third of all public expenditure in the UK.[284] On 24 February 2025, the Procurement Act 2023 took effect.[285] Previously, EU-based laws such as

In [ ]:
# !pip install --no-cache-dir recordclass

In [ ]:
# !pip install ragas==0.1.9

In [52]:
from ragas.integrations.langchain import EvaluatorChain 
# from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
)

# create evaluation chains
faithfulness_chain   = EvaluatorChain(metric=faithfulness)
answer_rel_chain     = EvaluatorChain(metric=answer_relevancy)
context_rel_chain    = EvaluatorChain(metric=context_relevancy)
context_recall_chain = EvaluatorChain(metric=context_recall)

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [53]:
# Recheck the result that we are going to validate.
result

{'query': 'What percentage of GDP did public procurement account for in OECD countries in 2019?',
 'result': 'In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.',
 'source_documents': [Document(id='d287777d-9ad8-4e85-9cec-c3afc8138931', metadata={'source': 'https://en.wikipedia.org/wiki/Public_procurement', 'title': 'Government procurement - Wikipedia', 'language': 'en'}, page_content='Government procurement or public procurement is the purchase of goods, works (construction), or services by the state, such as by a government agency or a state-owned enterprise.[1][2][3] In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.[4][5] In 2021, the World Bank Group estimated that public procurement made up about 15% of global GDP.[6] Therefore, government procurement accounts for a substantial part of the global economy.'),
  Document(id='213a9e87-da6c-4a9e-b5aa-8956c929d745', metadata={'source': 'https://en.wikipedia.org/wiki

In [56]:
import nest_asyncio
nest_asyncio.apply()

**Faithfulness**

In [60]:
eval_result = faithfulness_chain(result_updated)
eval_result["faithfulness"]

1.0

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [63]:
fake_result = result_updated.copy()
fake_result["result"] = "we are the champions"
eval_result = faithfulness_chain(fake_result)
eval_result["faithfulness"]

1.0

**Context Relevancy**

In [66]:
result_updated["ground_truth"] = eval_answers[1]  # matches question index used for result
eval_result = context_recall_chain(result_updated)
eval_result["context_recall"]

1.0

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [69]:
from langchain.schema import Document

fake_result = result_updated.copy()
fake_result["contexts"] = [Document(page_content="I love christmas").page_content]
eval_result = context_recall_chain(fake_result)
eval_result["context_recall"]

0.0

2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [71]:
predictions = qa_chain.batch(examples)

print("evaluating...")
r_faithfulness = []
for example, prediction in zip(examples, predictions):
    combined = {
        "question": example["query"],
        "answer": prediction["result"],
        "contexts": [doc.page_content for doc in prediction["source_documents"]]
    }
    score = faithfulness_chain(combined)
    r_faithfulness.append(score)
r_faithfulness

evaluating...


[{'question': 'What is government/public procurement?',
  'answer': 'Government procurement or public procurement is the process by which the government purchases goods, works (like construction), or services from private companies or organizations. This is done to provide public goods and merit goods to society, such as education, healthcare, infrastructure, and more. Government procurement is a significant part of the global economy, accounting for a substantial portion of GDP in many countries. It also plays a role in promoting innovation, economic growth, and competition within the private sector.',
  'contexts': ['Government procurement or public procurement is the purchase of goods, works (construction), or services by the state, such as by a government agency or a state-owned enterprise.[1][2][3] In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.[4][5] In 2021, the World Bank Group estimated that public procurement made up about 15% of global G

In [72]:
print("evaluating...")
r_recall = []
for example, prediction in zip(examples, predictions):
    combined = {
        "question": example["query"],
        "contexts": [doc.page_content for doc in prediction["source_documents"]],
        "ground_truth": example["ground_truths"][0]
    }
    score = context_recall_chain(combined)
    r_recall.append(score)
r_recall

evaluating...


[{'question': 'What is government/public procurement?',
  'contexts': ['Government procurement or public procurement is the purchase of goods, works (construction), or services by the state, such as by a government agency or a state-owned enterprise.[1][2][3] In 2019, public procurement accounted for approximately 12% of GDP in OECD countries.[4][5] In 2021, the World Bank Group estimated that public procurement made up about 15% of global GDP.[6] Therefore, government procurement accounts for a substantial part of the global economy.',
   'In addition to public goods, governments often also provide merit goods, such as education or health care. Merit goods are private goods which are rival and excludable and are therefore provided by private markets. Nevertheless, governments also provide merit goods because of reasons of equity and fairness and because they have positive externalities for society as a whole.[19] In order to provide public and merit goods, the government has to buy in